In [1]:
import sys

import matplotlib
import numpy
import pandas
import sklearn

print(sys.version.split()[0], numpy.__version__, pandas.__version__, sklearn.__version__, matplotlib.__version__)

3.12.5 2.3.4 2.3.3 1.7.2 3.10.7


In [2]:
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold, train_test_split

SEED = 3194          # last 4 digits of my student ID
data = pd.read_csv("data/spambase.csv")
feature_names = [c for c in data.columns if c != "spam"]
X, y = data[feature_names].to_numpy(), data["spam"].to_numpy()
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=SEED)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
print(X.shape, round(100 * y.mean(), 1), X_train.shape, X_test.shape)

(4204, 57) 39.9 (3153, 57) (1051, 57)


**SEED = 3194** (last 4 digits of my student ID)

# Task 4 — Random Forests & Ensembles (Smakov)

## Fixed models

In [3]:
from sklearn.base import clone
from sklearn.ensemble import AdaBoostClassifier, BaggingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import FunctionTransformer, StandardScaler
from sklearn.tree import DecisionTreeClassifier

tree = DecisionTreeClassifier(random_state=SEED)
bag = BaggingClassifier(DecisionTreeClassifier(), n_estimators=100, oob_score=True, random_state=SEED)
rf = RandomForestClassifier(n_estimators=300, oob_score=True, random_state=SEED, n_jobs=-1)
lr = make_pipeline(FunctionTransformer(np.log1p), StandardScaler(), LogisticRegression(max_iter=2000))
bag_lr = BaggingClassifier(lr, n_estimators=50, random_state=SEED)
knn = make_pipeline(FunctionTransformer(np.log1p), StandardScaler(), KNeighborsClassifier(n_neighbors=15))
ada = AdaBoostClassifier(n_estimators=400, random_state=SEED)
members = [("lr", lr), ("knn", knn), ("tree", tree)]   # members of voting and stacking in Parts 3-4
print("models defined")

models defined


## Part 1
Bagging and out-of-bag estimates

In [4]:
def oob_mask(sample_indices, n_samples):
    """Boolean mask of the rows that do NOT occur in sample_indices."""
    import numpy as np
    mask = np.ones(int(n_samples), dtype=bool)
    idx = np.asarray(sample_indices, dtype=int).ravel()
    if idx.size > 0:
        mask[idx] = False
    return mask

In [5]:
assert oob_mask([0, 2, 2], 5).tolist() == [False, True, False, True, True]
assert oob_mask([], 3).tolist() == [True, True, True]
assert oob_mask(np.array([3, 0]), 4).tolist() == [False, True, True, False]
assert oob_mask([0, 2, 2], 5).dtype == bool and oob_mask([], 3).shape == (3,)
print("oob_mask: all examples pass")

oob_mask: all examples pass


In [6]:
bag.fit(X_train, y_train)
n = X_train.shape[0]

oob_shares = np.array([oob_mask(s, n).mean() for s in bag.estimators_samples_])
mean_oob = oob_shares.mean()
p_not_drawn = (1 - 1 / n) ** n

print(f"n (training rows)            = {n}")
print(f"mean out-of-bag share        = {mean_oob:.4f}")
print(f"mean in-bag share            = {1 - mean_oob:.4f}")
print(f"formula (1 - 1/n)^n          = {p_not_drawn:.4f}")
print(f"formula 1 - (1 - 1/n)^n      = {1 - p_not_drawn:.4f}")

n (training rows)            = 3153
mean out-of-bag share        = 0.3675
mean in-bag share            = 0.6325
formula (1 - 1/n)^n          = 0.3678
formula 1 - (1 - 1/n)^n      = 0.6322


In [7]:
votes = np.zeros((n, 2))
for est, s in zip(bag.estimators_, bag.estimators_samples_):
    m = oob_mask(s, n)
    pred = est.predict(X_train[m]).astype(int)
    votes[np.where(m)[0], pred] += 1

oob_pred = np.argmax(votes, axis=1)          # tie -> class 0
oob_acc = (oob_pred == y_train).mean()
print(f"rows with no OOB vote: {(votes.sum(axis=1) == 0).sum()}")
print(f"OOB accuracy by hand : {oob_acc:.6f}")
print(f"bag.oob_score_       : {bag.oob_score_:.6f}")

rows with no OOB vote: 0
OOB accuracy by hand : 0.933714
bag.oob_score_       : 0.933714


In [8]:
from sklearn.model_selection import cross_validate

all_scores = {}   # fold scores of every model on the same cv folds (reused in Parts 3, 4, 6)

def cv_scores(name, model):
    s = cross_validate(model, X_train, y_train, cv=cv, scoring="accuracy")["test_score"]
    all_scores[name] = s
    return s

def cv_table(names):
    rows = {}
    for nm in names:
        s = all_scores[nm]
        rows[nm] = list(s) + [s.mean(), np.std(s)]
    cols = [f"fold {i+1}" for i in range(5)] + ["mean", "std"]
    return pd.DataFrame.from_dict(rows, orient="index", columns=cols).round(4)

def paired_table(pairs):
    rows = {}
    for a, b in pairs:
        d = all_scores[a] - all_scores[b]
        m, s = d.mean(), np.std(d)
        rows[f"{a} - {b}"] = list(d) + [m, s, "inside spread" if abs(m) <= s else "outside spread"]
    cols = [f"fold {i+1}" for i in range(5)] + ["mean", "std", "verdict"]
    df = pd.DataFrame.from_dict(rows, orient="index", columns=cols)
    num = cols[:7]
    df[num] = df[num].astype(float).round(4)
    return df

In [9]:
for name, model in [("tree", tree), ("bag", bag), ("rf", rf), ("lr", lr), ("bag_lr", bag_lr)]:
    cv_scores(name, model)

display(cv_table(["tree", "bag", "rf", "lr", "bag_lr"]))
display(paired_table([("bag", "tree"), ("rf", "tree"), ("bag_lr", "lr")]))

,fold 1,fold 2,fold 3,fold 4,fold 5,mean,std
tree,0.9128,0.9113,0.9113,0.8889,0.8778,0.9004,0.0144
bag,0.9382,0.9319,0.9493,0.9127,0.9238,0.9312,0.0124
rf,0.9525,0.9334,0.9620,0.9317,0.9397,0.9439,0.0116
lr,0.9239,0.9239,0.9461,0.9206,0.9365,0.9302,0.0096
bag_lr,0.9239,0.9239,0.9461,0.9238,0.9365,0.9309,0.0091


,fold 1,fold 2,fold 3,fold 4,fold 5,mean,std,verdict
bag - tree,0.0254,0.0206,0.0380,0.0238,0.0460,0.0308,0.0097,outside spread
rf - tree,0.0396,0.0222,0.0507,0.0429,0.0619,0.0435,0.0131,outside spread
bag_lr - lr,0.0000,0.0000,0.0000,0.0032,0.0000,0.0006,0.0013,inside spread


In [10]:
rf.fit(X_train, y_train)
n_test = len(y_test)
p_rf = rf.score(X_test, y_test)
se_rf = np.sqrt(p_rf * (1 - p_rf) / n_test)
rf_cv = all_scores["rf"]

print(f"rf.oob_score_      = {rf.oob_score_:.4f}")
print(f"rf CV mean (step 3) = {rf_cv.mean():.4f}   (fold std = {np.std(rf_cv):.4f})")
print(f"rf test accuracy p  = {p_rf:.4f}   SE = {se_rf:.4f}   (p ± SE: {p_rf - se_rf:.4f} … {p_rf + se_rf:.4f})")
print(f"|oob - cv mean|     = {abs(rf.oob_score_ - rf_cv.mean()):.4f}  -> "
      f"{'agree' if abs(rf.oob_score_ - rf_cv.mean()) <= np.std(rf_cv) else 'do not agree'}")

rf.oob_score_      = 0.9445
rf CV mean (step 3) = 0.9439   (fold std = 0.0116)
rf test accuracy p  = 0.9553   SE = 0.0064   (p ± SE: 0.9489 … 0.9617)
|oob - cv mean|     = 0.0006  -> agree


### Part 1 results

**OOB share.** There are n = 3153 training rows. The mean OOB share over 100 trees is 0.3675 and the in-bag share is 0.6325. The formula gives (1 - 1/n)^n = 0.3678 and 1 - (1 - 1/n)^n = 0.6322, so my numbers are very close to it. Each tree does not see about 37% of the rows.

**OOB accuracy by hand.** Every row got at least one OOB vote. My OOB accuracy is 0.933714 and bag.oob_score_ is 0.933714, they are the same.

**(a)** For the tree, bag - tree = 0.0308 ± 0.0097. Bagging was better on all 5 folds and |mean| > std, so the difference is outside the fold spread and bagging improves the tree (0.9004 -> 0.9312). Random forest is even better, rf - tree = 0.0435 ± 0.0131, also outside the spread. The reason is that a full decision tree has high variance: if the training rows change a little, the tree changes a lot. Bagging trains many trees on different bootstrap samples and takes the vote, so the errors of single trees partly cancel and the variance goes down. Random forest also picks random features at each split, so the trees are less similar and averaging works even better.

For logistic regression, bag_lr - lr = 0.0006 ± 0.0013, and on 4 folds the difference is exactly 0. |mean| <= std, so it is inside the fold spread and my data cannot show that bagging changes logistic regression. Logistic regression is a stable model with low variance, so the 50 bootstrap models are almost the same as the original one and averaging them gives almost the same predictions. Bagging reduces variance, not bias, so it helps unstable models like trees but not this one.

**(b)** rf.oob_score_ = 0.9445 and the CV mean = 0.9439. The difference is 0.0006, smaller than the std of the rf fold scores (0.0116), so by the task rule they agree and OOB is a fair replacement for CV here. It also needs no extra fitting. The test accuracy is p = 0.9553 with SE = 0.0064. Both OOB and CV are a bit lower than the test accuracy, by about 0.011 (around 1.7 SE). This gap is smaller than the fold std, so it can be just the noise of one test split. OOB and CV are also expected to be a little pessimistic: in OOB each row is predicted by only about 37% of the trees, and in CV each model is trained on 80% of the training rows, while the final forest uses all 3153 rows.